In [25]:
import osmium
import geopandas
import numpy as np
import csv
import matplotlib.pyplot as plt

def haversine_np(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    Reference:
        https://stackoverflow.com/a/29546836/7657658
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(
        dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2

    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c
    return km

fp = osmium.FileProcessor("cataluna-latest.osm.pbf")\
        .with_locations()\
        .with_filter(osmium.filter.EmptyTagFilter())\
        .with_filter(osmium.filter.EntityFilter(osmium.osm.WAY))\
        .with_filter(osmium.filter.KeyFilter('maxspeed'))\
        .with_filter(osmium.filter.GeoInterfaceFilter())

nodes = {}
edges = [[]]

for obj in fp:
    for idx, node in enumerate(obj.nodes):
        if idx == 0:
            continue
        edges.append([str(obj.nodes[idx - 1].ref), str(obj.nodes[idx].ref), haversine_np(obj.nodes[idx - 1].lat, obj.nodes[idx - 1].lon, obj.nodes[idx].lat, obj.nodes[idx].lon)])

# nodes = np.array(nodes)
# plt.plot(nodes[:,1], nodes[:,2])

with open("nodes.csv", "w") as csvfile:
    writer = csv.writer(csvfile)
    for node in nodes:
        writer.writerow(node)

with open("edges.csv", "w") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(edges)

